# Laboratorul 7: Agentic AI - Agenti

## 1.) Intro: Services and Inter-connected Processes

Services communicate one with another through requests.

This whole situation can be views as 2 entities: a CLIENT and a SERVER talking one to another.


```
                  ┌────────────────────────────────────────────────────────┐
                  │                      HOST / CLIENT                     │
                  │                     wants something                    │
                  └───────────────────────────┬────────────────────────────┘
                                              │
                                  makes a request to a server
                                              │
                                              ▼
                  ┌────────────────────────────────────────────────────────┐
                  │                       SERVER                           │
                  │                     does the work                      │
                  └───────────────────────────┬────────────────────────────┘
                                              │
                                     reports to client
                                              │
                                              ▼
                  ┌────────────────────────────────────────────────────────┐
                  │                     HOST / CLIENT                      │
                  │                     response/error                     │
                  └────────────────────────────────────────────────────────┘
```

Most important Status Codes:

### 1. HTTP Status Codes: The Language of Web Services

When services communicate, they use **HTTP Status Codes** to quickly convey the outcome of a request. These are divided into five standard classes:

| Class | Type | Description | Key Examples for AI/Agents |
| :--- | :--- | :--- | :--- |
| **1xx** | Informational | Request received, continuing process. | `100 Continue`, `101 Switching Protocols` |
| **2xx** | Success | The action was successfully received, understood, and accepted. | `200 OK`, `201 Created` (resource created), `204 No Content` |
| **3xx** | Redirection | Further action needs to be taken to complete the request. | `301 Moved Permanently`, `304 Not Modified` (caching) |
| **4xx** | Client Error | The request contains bad syntax or cannot be fulfilled. | `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `429 Too Many Requests` |
| **5xx** | Server Error | The server failed to fulfill an apparently valid request. | `500 Internal Server Error`, `502 Bad Gateway`, `503 Service Unavailable` |

---

#### Very important Client Errors (4xx):
*   **`400 Bad Request`**: The server cannot process the request due to client error (e.g., malformed request syntax, invalid parameters). *In Agentic AI, this often happens if an LLM generates invalid arguments for a tool.*
*   **`401 Unauthorized`**: The request requires user authentication. You lack valid credentials (API keys, tokens).
*   **`403 Forbidden`**: The server understood the request, but refuses to authorize it. Even with credentials, you don't have permission for this resource.
*   **`404 Not Found`**: The requested resource could not be found. *For agents, this might mean a requested database record or API endpoint doesn't exist.*
*   **`429 Too Many Requests`**: Rate limiting is active. The client has sent too many requests in a given amount of time. *Crucial for LLM agents to handle gracefully with exponential backoff.*

#### Very important Server Errors (5xx):
*   **`500 Internal Server Error`**: A generic error message when the server encounters an unexpected condition.
*   **`502 Bad Gateway`**: The server, while acting as a gateway or proxy, received an invalid response from the upstream server.
*   **`503 Service Unavailable`**: The server is temporarily unable to handle the request (due to overloading or maintenance).
*   **`504 Gateway Timeout`**: The server, while acting as a gateway or proxy, did not receive a timely response from the upstream server.


### 2. Communication Protocols: From REST to JSON-RPC

While standard web services often use **REST (Representational State Transfer)** APIs with HTTP methods (GET, POST, PUT, DELETE), many agentic systems and real-time protocols (like MCP) rely on **JSON-RPC**.

#### REST vs. JSON-RPC: A Quick Comparison

*   **REST (Resource-Oriented)**:
    *   Focuses on **resources** (e.g., `/users`, `/tools`) and manipulates them using standard HTTP verbs.
    *   *Example*: `GET /tools/calculator` to fetch details, `POST /tools/calculator/run` to execute.
*   **JSON-RPC (Action-Oriented)**:
    *   A lightweight Remote Procedure Call (RPC) protocol encoded in JSON.
    *   Focuses on **executing methods/functions** on a remote server.
    *   Uses a single transport endpoint (usually over standard I/O or a single HTTP POST endpoint).
    *   *Example request*:
        ```json
        {
          "jsonrpc": "2.0",
          "method": "tools/call",
          "params": {
            "name": "calculate",
            "arguments": { "expression": "2 + 2" }
          },
          "id": 1
        }
        ```

#### The JSON-RPC Message Types
1.  **Request**: Client asks the server to run a method. Expects a **Response** (either `result` or `error`) with a matching `id`.
2.  **Notification**: A one-way message (no `id` field). Sent without expecting a response (e.g., status updates, logs).


### 3. Model Context Protocol (MCP): The Client-Server Model for AI

The **Model Context Protocol (MCP)** is an open standard that brings the classic Client-Server architecture directly to AI applications. Before building or working with MCP servers, tools, and agents, we must understand how they interconnect.

```
                  ┌────────────────────────────────────────────────────────┐
                  │                      HOST / CLIENT                     │
                  │  (e.g., Cursor IDE, Claude Desktop, Custom AI Agent)   │
                  └───────────────────────────┬────────────────────────────┘
                                              │
                                  Exchanges JSON-RPC messages
                                  via Stdio or SSE transport
                                              │
                                              ▼
                  ┌────────────────────────────────────────────────────────┐
                  │                       MCP SERVER                       │
                  │  (Exposes Tools, Resources, and Prompts to the Client) │
                  └───────────────────────────┬────────────────────────────┘
                                              │
                                     Executes local/remote
                                     code & integrations
                                              │
                                              ▼
                  ┌────────────────────────────────────────────────────────┐
                  │                 EXTERNAL DATA & TOOLS                  │
                  │  (e.g., Postgres DB, GitHub API, Local Filesystem)     │
                  └────────────────────────────────────────────────────────┘
```

#### Essential Concepts to Know Before Working with MCP

1.  **What are tools?**:
    *   **Tools**: Executable functions that the AI agent can discover and run (e.g., `execute_sql`, `read_file`, `fetch_webpage`). On the other way, tools can also be views as: **Resources**: Read-only data sources that the AI can inspect (e.g., database schemas, file contents, API documentation).

2.  **Transport Mechanisms (How they connect)**:
    *   **Stdio Transport (Standard Input/Output)**:
        *   The client spawns the MCP server as a **local subprocess**.
        *   Communication happens directly over `stdin` and `stdout`.
    *   **SSE Transport (Server-Sent Events)**:
        *   The client connects to a remote or local hoested MCP server running over HTTP.
        *   The server streams events to the client using SSE, and the client sends messages back via HTTP POST.


## 2.) Tools

We can build tools in three ways:
- plain Python functions, with no server
- one MCP server with tools
- multiple MCP servers mounted on different paths

**FastMCP**
FastMCP is a high-level framework for implementing MCP servers and clients.
- Protocol handling (JSON-RPC, sessions)
- Transport support (stdio, HTTP, SSE)

VERY IMPORTANT FOR A TOOL IS:
- **tool's description**
- **tool's name**
- **tool's paramters**
- **tool's paramters names**

Tools let an agent interact with files, the web, databases, and other services.

A useful split is:
- read tools: inspect or fetch information
- write tools: change state, like saving a note or adding a calendar event

We have implemented these tools:
- `read_schedule()`: reads the class schedule CSV and can filter by day.
- `search_notes()`: scans Markdown notes and groups matches by file.
- `grep_pdf_chapter()`: looks through the first PDF pages and returns matching paragraphs with page numbers.
- `get_weather()`: asks an external service for weather in a chosen location.
- `web_search()`: searches the web and returns a short result list.
- `add_calendar_event()`: appends a new event to the calendar.
- `save_study_note()`: creates a new .md note file in the notes directory.

Debugging browser tool used for debugging mcp tools:

- MCP Inspector: npx @modelcontextprotocol/inspector

## 3.) Models

### Why abstraction helps

The agent code does not need to change when the model changes. That makes it easy to compare a smaller model with a stronger one using the same prompt, the same tools, and the same agent logic.

There are different models providers, such as:
- OpenAi
- AzureOpenAi
- Anthropic


In order to use a mode, we usually need some details about it, such as:
- deployment_name
- deployment_date
- api_key !!!!! THIS SHOULD NEVER BE REVEALED TO PUBLIC SOURCES
- other request_paramters
- ...

```python
from agno.agent import Agent
from section_3.models import get_model, list_models

print(list_models())

model = get_model('balanced')
agent = Agent(model=model, markdown=True)
agent.print_response('Say hello in one short sentence.')
```

If you change `balanced` to `fast` or `strong`, the script stays the same. Only the model behind the alias changes.

### Request parameters

Some extra parameters are passed with the request, not with the model identity. A few useful ones are:
- `top_p`: controls how broad the sampling is
- `frequency_penalty`: reduces repeated wording
- `presence_penalty`: encourages the model to bring in new ideas
- `stop`: tells the model when to stop
- `metadata`: adds small extra tags for tracing or debugging

## 4.) Agents

An agent can have: Normal tools AND Predefined tools such as Reasoning Tools and Caching Tools.

An Agent is a specialised LLM that can use tools and be tunned for a specific task

We will use **Agno** as framework. But there are many other agentic frameworks that are really powerful, such as: google adk, crewai, mastra.

Agno documentation: https://docs.agno.com/

There is the Agent class. This class has many fields that can be set, but some of them are extremely important in practice:
- model
- name
- id
- enable_session_summaries: bool = False
- add_session_summary_to_context: Optional[bool] = None
- add_history_to_context: bool = False
- num_history_runs: Optional[int] = None
- tools
- description
- instructions
- role
- expected_output
...

### In practice

An Agent usually follows this loop:
- read the user request
- decide whether it can answer directly or should call a tool
- use the tool if needed
- turn the tool result into a clean final answer

Two things matter most in practice:
- `instructions`: tell the agent when to use each tool
- `tools`: give the agent real actions instead of hallucinating the answer

When we want memory across runs, we also add history or session summaries, but only if the agent has a DB or session store behind it.

### Personal Assistant Agent

An Agent that can:
- read the schedule of a user
- get weather details
- search things on web
- add an event to the calendar

### College Buddy Assistent Agent

An Agent that can:
- search for something in a directory with notes
- search different things in a pdf
- write and save a note based on other requests

## 5.) Teams

There is the Team class.

A Team is essencially an Agent, with a very important nuance. A Team is an Agent that has other Agents/Teams under it's command.

### In practice

A Team adds coordination on top of agents:
- the team leader decides which member should handle each part of the request
- members can run separately and return partial answers
- the leader combines the member outputs into one response

Useful team settings here are:
- `determine_input_for_members`: helps the leader send the right task to the right member
- `show_members_responses`: lets us see what each subagent returned
- `store_member_responses` and `store_events`: keep the trace for debugging and for learning from previous runs

For our lab, the team leader delegates personal tasks to the Personal Assistant Agent and study/document tasks to the College Buddy Agent.

Very important fields here, other that the fields that an Agent has, are:
- mode: Optional[TeamMode] = None
- respond_directly: bool = False
- delegate_to_all_members: bool = False
- determine_input_for_members: bool = True

### Team coordinator



This will be a Team, which has 2 subagents that he will coorinate:
- Personal Assistent Agent
- College Buddy Assistent Agent